# Notebook 2 — Janelas Deslizantes e Pipeline de Streaming

## Aula 7: Otimizações e Escala no Monitoramento de Drift

### Objetivos

1. Implementar pipeline de monitoramento de drift com **janelas deslizantes** (sliding windows).
2. Calcular **PSI incremental** em janelas, sem recomputar do zero a cada passo.
3. Simular **pipeline de streaming em lote** (micro-batches) emulando Kafka + Flink.
4. Comparar resultados com janelas de tamanhos diferentes (trade-off sensibilidade vs. ruído).
5. Compreender as arquiteturas **Lambda vs. Kappa** para monitoramento em escala.

### Conexão com o Documento 04

> *"Monitorar drift em streaming implica escolher uma política de janela. Janelas curtas
> aumentam sensibilidade e reduzem atraso de detecção, mas podem produzir estimativas
> ruidosas. Janelas longas estabilizam estimativas, mas podem esconder drifts súbitos."*
> — DOCUMENTO_AULA_7.md, seção 'Saiba Mais'

### Vídeos Relacionados

- **Vídeo 7.2**: Arquiteturas Lambda e Kappa para streaming de drift.
- **Vídeo 7.3**: Monitoramento de drift com amostragem e janelas deslizantes.

In [ ]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adicionar diretório pai ao path para importar src/
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.model import DriftMonitor, IncrementalPSI
from src.training import (
    train_model,
    train_sliding_window_pipeline,
    simulate_streaming_pipeline,
)
from src.evaluation import plot_psi_timeline, plot_drift_heatmap
from src.utils import save_model, save_metrics, emit_alert

# Configurações de visualização
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

%matplotlib inline

## 1. Carregar e Preparar Dados

Carregamos o dataset FinBank e preparamos baseline para monitoramento.

In [ ]:
# Carregar dataset
preprocessor = DataPreprocessor(random_state=42)
df = preprocessor.load_data("../data/raw/finbank_transactions.csv")
df = preprocessor.clean_data(df)

feature_columns = DataPreprocessor.NUMERIC_FEATURES
print(f"Dataset: {len(df)} transações")
print(f"Features monitoradas: {feature_columns}")
print(f"Meses disponíveis: {sorted(df['month'].unique())}")

## 2. Pipeline de Janelas Deslizantes (Sliding Windows)

Implementamos o pipeline de monitoramento conforme descrito na Videoaula 3.
A cada janela, comparamos a distribuição corrente com o baseline usando
PSI e K–S, e emitimos alertas conforme o Snippet 4 do Hands On:

```python
PSI_CRITICO = 0.25
ALFA = 0.05
drift = (psi_value >= PSI_CRITICO) or (p_value < ALFA)
```

### Janela de 1 mês (alta sensibilidade)

Conforme a seção 'Saiba Mais':
> *"Uma estratégia comum é adotar múltiplas escalas simultaneamente: uma janela curta
> para detecção rápida e uma janela longa para tendência e sazonalidade."*

In [ ]:
# Pipeline com janela de 1 mês
monitor = DriftMonitor(n_bins=10, psi_threshold=0.25, psi_warning=0.10, ks_alpha=0.05)

results_1m = train_sliding_window_pipeline(
    df, feature_columns, monitor,
    window_size=1, step_size=1,
    baseline_months=(1, 2, 3),
)

print(f"Janelas processadas: {len(results_1m)}\n")
for r in results_1m:
    months = r["window_months"]
    score = r["aggregate_score"]
    alert = r["alert_level"]
    retrain = r["should_retrain"]
    print(f"  Mês(es) {months}: PSI_médio={score['mean_psi']:.4f}, "
          f"Alerta={alert}, Retrain={retrain}")

In [ ]:
# Visualizar timeline de PSI — janela de 1 mês
fig = plot_psi_timeline(
    results_1m,
    psi_warning=0.10,
    psi_critical=0.25,
    title="PSI ao Longo do Tempo — Janela de 1 Mês",
    save_path="../outputs/figures/psi_timeline_1m.png",
)
plt.show()

### Janela de 2 meses (menor ruído)

Janelas mais longas estabilizam estimativas, reduzindo falsos alarmes,
porém podem atrasar a detecção de drifts súbitos.

In [ ]:
# Pipeline com janela de 2 meses
monitor_2m = DriftMonitor(n_bins=10, psi_threshold=0.25, psi_warning=0.10)

results_2m = train_sliding_window_pipeline(
    df, feature_columns, monitor_2m,
    window_size=2, step_size=1,
    baseline_months=(1, 2, 3),
)

print(f"Janelas processadas (2 meses): {len(results_2m)}\n")
for r in results_2m:
    print(f"  Mês(es) {r['window_months']}: PSI_médio={r['aggregate_score']['mean_psi']:.4f}, "
          f"Alerta={r['alert_level']}")

In [ ]:
# Comparar janelas de 1 mês vs. 2 meses
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (results, title) in zip(axes, [
    (results_1m, "Janela 1 Mês"),
    (results_2m, "Janela 2 Meses"),
]):
    psi_vals = [r["aggregate_score"]["mean_psi"] for r in results]
    labels = [f"M{'-'.join(str(m) for m in r['window_months'])}" for r in results]
    colors = ["red" if v >= 0.25 else "orange" if v >= 0.10 else "green" for v in psi_vals]

    ax.bar(range(len(psi_vals)), psi_vals, color=colors, alpha=0.8,
           edgecolor="black", linewidth=0.5)
    ax.axhline(y=0.10, color="orange", linestyle="--", label="Warning")
    ax.axhline(y=0.25, color="red", linestyle="--", label="Crítico")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(title)
    ax.set_ylabel("PSI Médio")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Comparação de Tamanhos de Janela — Trade-off Sensibilidade vs. Ruído",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/comparacao_janelas.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Heatmap de Drift por Feature e Janela

Visualização que apoia a priorização de monitoramento por criticidade
de negócio, conforme discutido na Videoaula 4.

In [ ]:
# Heatmap de PSI por feature e janela (1 mês)
fig = plot_drift_heatmap(
    results_1m,
    feature_columns=feature_columns,
    title="Heatmap de PSI por Feature e Janela (1 mês)",
    save_path="../outputs/figures/heatmap_drift_1m.png",
)
plt.show()

## 4. PSI Incremental — Cálculo em Janelas Deslizantes

Implementamos o cálculo incremental de PSI usando a classe `IncrementalPSI`.
Em vez de recomputar histogramas do zero, atualizamos contagens e removemos
dados antigos da janela, conforme descrito na seção 'Saiba Mais':

> *"O uso de estados agregados (contagens por bin, somas, quantis aproximados)
> permite computar PSI e outras métricas sem materializar o conjunto completo."*

In [ ]:
# Demonstrar PSI incremental
# 1. Definir bins a partir do baseline
baseline_df = df[df["month"].isin([1, 2, 3])]
feat = "amount"

ref_values = baseline_df[feat].values
bins = np.quantile(ref_values, np.linspace(0, 1, 11))  # 10 bins
bins = np.unique(bins)

# Proporções de referência
ref_counts, _ = np.histogram(ref_values, bins=bins)
ref_proportions = ref_counts / ref_counts.sum()

# 2. Calcular PSI incrementalmente por mês
inc_psi = IncrementalPSI(bins=bins)
incremental_results = []

for month in range(4, 9):
    month_data = df[df["month"] == month][feat].values

    # Resetar para janela não acumulativa
    inc_psi.reset()
    inc_psi.update(month_data)

    psi_val = inc_psi.compute(ref_proportions)
    incremental_results.append({"Mês": month, "PSI_incremental": psi_val})

inc_df = pd.DataFrame(incremental_results)
print(f"PSI Incremental para feature '{feat}':")
inc_df

## 5. Simulação de Pipeline de Streaming (Micro-batches)

Emulamos o comportamento de um pipeline orientado a eventos (Kafka + Flink)
processando transações em mini-batches, conforme descrito na Videoaula 2.

### Arquitetura Kappa (streaming unificado)

Conforme a Tabela 1 do DOCUMENTO_AULA_7.md:
> *"Kappa costuma ser atraente porque drift é, por definição, um fenômeno temporal
> e incremental: deseja-se um fluxo unificado de métricas em janelas, com baixo
> tempo de resposta."*

```
Transações → [Kafka topic] → [Flink/PyFlink] → Métricas PSI/KS → [Alertas]
                                    ↓
                             Janela deslizante
```

In [ ]:
# Simular streaming com micro-batches de 500 transações
monitor_stream = DriftMonitor(n_bins=10, psi_threshold=0.25, psi_warning=0.10)

# Callback de drift (simula trigger de retraining)
drift_alerts = []

def on_drift(result):
    alert = emit_alert(
        f"Drift detectado no batch {result['batch_idx']}: "
        f"PSI_max={result['aggregate_score'].get('max_psi', 0):.4f}",
        level=result["alert_level"],
    )
    drift_alerts.append(alert)

streaming_results = simulate_streaming_pipeline(
    df, feature_columns, monitor_stream,
    batch_size=500,
    baseline_months=(1, 2, 3),
    on_drift_callback=on_drift,
)

print(f"Batches processados: {len(streaming_results)}")
print(f"Alertas emitidos: {len(drift_alerts)}\n")

# Resumo por batch
for r in streaming_results:
    print(f"  Batch {r['batch_idx']:2d} | n={r['n_samples']:4d} | "
          f"Meses={r['months_in_batch']} | "
          f"PSI_médio={r['aggregate_score']['mean_psi']:.4f} | "
          f"Alerta={r['alert_level']}")

In [ ]:
# Visualizar PSI ao longo dos batches (simula streaming)
batch_psis = [r["aggregate_score"]["mean_psi"] for r in streaming_results]
batch_labels = [f"B{r['batch_idx']}" for r in streaming_results]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["red" if v >= 0.25 else "orange" if v >= 0.10 else "green"
          for v in batch_psis]
ax.bar(range(len(batch_psis)), batch_psis, color=colors, alpha=0.7,
       edgecolor="black", linewidth=0.3)
ax.axhline(y=0.10, color="orange", linestyle="--", label="Warning (0.10)")
ax.axhline(y=0.25, color="red", linestyle="--", label="Crítico (0.25)")
ax.set_xlabel("Batch")
ax.set_ylabel("PSI Médio")
ax.set_title("Simulação de Streaming — PSI por Micro-batch (batch_size=500)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/streaming_psi.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Treinamento do Modelo de Fraude (Baseline)

Treinamos um classificador de fraude com dados do baseline para
avaliar no próximo notebook como o drift degrada seu desempenho.

In [ ]:
# Preparar features para treinamento
baseline_df = df[df["month"].isin([1, 2, 3])].copy()
baseline_prepared = preprocessor.prepare_features(baseline_df, encode_categorical=True)

# Selecionar features numéricas + dummies
train_features = [
    c for c in baseline_prepared.columns
    if c not in ["transaction_id", "timestamp", "month", "is_fraud"]
]

X_train = baseline_prepared[train_features].values
y_train = baseline_prepared["is_fraud"].values

print(f"Features de treinamento: {len(train_features)}")
print(f"X_train: {X_train.shape}")
print(f"Taxa de fraude (baseline): {y_train.mean():.2%}")

In [ ]:
# Treinar modelo de detecção de fraude
clf = train_model(X_train, y_train)
print(f"Modelo treinado: {type(clf).__name__}")
print(f"Acurácia no treino: {clf.score(X_train, y_train):.4f}")

# Salvar modelo para uso no Notebook 03
save_model(clf, "../outputs/models/fraud_classifier.pkl")
print("\nModelo salvo em outputs/models/fraud_classifier.pkl")

# Salvar lista de features usadas
save_metrics(
    {"train_features": train_features, "n_train": len(X_train)},
    "../outputs/models/train_config.json",
)

## 7. Arquiteturas Lambda vs. Kappa — Comparação

Conforme a Tabela 1 do DOCUMENTO_AULA_7.md:

| Característica | Lambda | Kappa |
|---|---|---|
| Camadas | Batch + Streaming | Streaming unificado |
| Latência | Variável | Baixa e consistente |
| Complexidade | Maior (lógica duplicada) | Menor (replay) |
| Fit para drift | Auditoria histórica mandatória | Monitoramento contínuo e simplicidade |

> *"No caso FinBank, a Kappa costuma ser atraente porque drift é, por definição,
> um fenômeno temporal e incremental."*
> — DOCUMENTO_AULA_7.md, seção 'Lambda vs. Kappa'

In [ ]:
# Salvar métricas de drift para análise futura
save_metrics(
    {
        "pipeline": "sliding_window_1m",
        "n_windows": len(results_1m),
        "results": [
            {
                "window_months": r["window_months"],
                "aggregate_score": r["aggregate_score"],
                "alert_level": r["alert_level"],
                "should_retrain": r["should_retrain"],
            }
            for r in results_1m
        ],
    },
    "../outputs/logs/drift_results_1m.json",
)
print("Métricas salvas em outputs/logs/drift_results_1m.json")

## Resumo

Neste notebook:

1. **Implementamos** pipeline de monitoramento com janelas deslizantes de 1 e 2 meses.
2. **Calculamos** PSI incremental usando a classe `IncrementalPSI`.
3. **Simulamos** pipeline de streaming em micro-batches (emulando Kafka + Flink).
4. **Comparamos** tamanhos de janela e seu impacto na detecção de drift.
5. **Treinamos** modelo de detecção de fraude com dados do baseline.
6. **Discutimos** as arquiteturas Lambda vs. Kappa para monitoramento em escala.

### Próximo Notebook

No **Notebook 03 — Avaliação**, avaliaremos a qualidade da detecção de drift,
analisaremos o impacto do drift no desempenho do modelo e implementaremos
triggers automáticos de retraining.